# Frequency Shifting and Audio Mixing

Explore partial spectrum shifts, frequency-domain mixing, and inverse shifting using two audio tracks.

Run the cells in order. This notebook is independent of the other notebooks.


In [ ]:
from pathlib import Path

# Resolve paths when launched from the repository root or notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "audio"
OUTPUT_DIR = ROOT / "outputs"
if not DATA_DIR.is_dir():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Incomplete spectrum shift


In [ ]:
import numpy as np
from scipy.io import wavfile
from IPython.display import Audio, display

# ---------------------------- Helpers ----------------------------------------
def to_mono_float(x: np.ndarray) -> np.ndarray:
    """Convert stereo/int arrays to mono float32 in [-1, 1], keeping behavior."""
    if x.ndim == 2:
        x = x.mean(axis=1)
    if np.issubdtype(x.dtype, np.integer):
        x = x.astype(np.float32) / np.iinfo(x.dtype).max
    else:
        x = x.astype(np.float32)
    return x

def load_wav_as_mono(path: str) -> tuple[int, np.ndarray]:
    """Read a WAV and return (fs, mono_float_signal)."""
    fs, x = wavfile.read(path)
    return fs, to_mono_float(x)

def fft_signal(x: np.ndarray) -> np.ndarray:
    """1D FFT wrapper (kept for clarity/extensibility)."""
    return np.fft.fft(x)

def ifft_real(X: np.ndarray) -> np.ndarray:
    """IFFT then take real part (matches original behavior)."""
    return np.real(np.fft.ifft(X))

def incomplete_positive_shift(X: np.ndarray, k_bins: int) -> np.ndarray:
    """
    Reproduce the original 'incomplete frequency shift':
    Roll only indices N//2 .. N-1 by k bins (the upper half of the unshifted FFT).
    """
    N = X.shape[0]
    X_shifted = X.copy()
    pos_half = np.arange(N // 2, N)
    X_shifted[pos_half] = np.roll(X_shifted[pos_half], k_bins)
    return X_shifted

def bins_for_hz(f_shift_hz: float, N: int, fs: int) -> int:
    """Match the original rounding from Hz to FFT-bin shift."""
    return int(round(f_shift_hz * N / fs))

def play(label: str, x: np.ndarray, fs: int) -> None:
    print(label)
    display(Audio(x, rate=fs))

# ---------------------------- Core pipeline ----------------------------------
class FrequencyShiftExperiment:
    def __init__(self, file1: str, file2: str, f_shift_hz: float = 5000.0):
        self.file1 = file1
        self.file2 = file2
        self.f_shift_hz = f_shift_hz
        self.fs = None
        self.x1 = None
        self.x2 = None

    def load_inputs(self) -> None:
        fs1, x1 = load_wav_as_mono(self.file1)
        fs2, x2 = load_wav_as_mono(self.file2)
        assert fs1 == fs2, "The sampling rate of the two files must be the same."
        self.fs = fs1
        self.x1, self.x2 = x1, x2

    def demo_listen_originals(self) -> None:
        play("music1:", self.x1, self.fs)
        play("music2:", self.x2, self.fs)

    def make_incomplete_shift(self) -> np.ndarray:
        X1 = fft_signal(self.x1)
        _ = fft_signal(self.x2)

        N = len(self.x1)
        k = bins_for_hz(self.f_shift_hz, N, self.fs)
        X1_bad = incomplete_positive_shift(X1, k)
        x1_bad = ifft_real(X1_bad)
        return x1_bad

# ---------------------------- Run --------------------------------------------
print("Incomplete spectrum shift:")
exp = FrequencyShiftExperiment((DATA_DIR / "music1.wav"), (DATA_DIR / "music2.wav"), f_shift_hz=5000.0)
exp.load_inputs()
exp.demo_listen_originals()

x1_bad = exp.make_incomplete_shift()
play("music1 with incomplete frequency shift (Unclear)", x1_bad, exp.fs)
# -----------------------------------------------------------------------------

## Frequency-domain mixing


In [ ]:
# ---------------------------- New helpers -------------------------------------
def align_lengths(xa: np.ndarray, xb: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Truncate both signals to the same (minimum) length — identical to original behavior."""
    L = min(len(xa), len(xb))
    return xa[:L], xb[:L]

def shift_in_centered_spectrum(S: np.ndarray, k_bins: int) -> np.ndarray:
    """
    Frequency-domain shift exactly as in the original mixing experiment:
    1) fftshift to center DC,
    2) roll by k bins,
    3) ifftshift back.
    """
    S_center = np.fft.fftshift(S)
    S_center = np.roll(S_center, k_bins)
    return np.fft.ifftshift(S_center)

# ---------------------------- mixing experiment logic ------------------------------------
class FrequencyDomainMixer:
    """
    Mixes two signals by shifting the spectrum of track A by k bins
    (computed from desired f_shift_hz) and summing spectra with track B.
    """
    def __init__(self, fs: int, f_shift_hz: float = 5000.0):
        self.fs = fs
        self.f_shift_hz = f_shift_hz

    def mix(self, x1: np.ndarray, x2: np.ndarray) -> np.ndarray:
        # 1) Match lengths (truncate)
        a, b = align_lengths(x1, x2)

        # 2) Compute shift amount in bins for the matched length
        N = len(a)
        k = bins_for_hz(self.f_shift_hz, N, self.fs)

        # 3) FFTs
        S1 = fft_signal(a)
        S2 = fft_signal(b)

        # 4) Shift S1 in centered spectrum (fftshift → roll → ifftshift)
        S1_shifted = shift_in_centered_spectrum(S1, k)

        # 5) Sum in frequency domain and return real IFFT
        S_mix = S1_shifted + S2
        x_mix = ifft_real(S_mix)
        return x_mix

# ---------------------------- Usage -------------------------------------------
print("Frequency-domain mixture:")
exp = FrequencyShiftExperiment((DATA_DIR / "music1.wav"), (DATA_DIR / "music2.wav"), f_shift_hz=5000.0); exp.load_inputs()
x1, x2, fs = exp.x1, exp.x2, exp.fs
mixer = FrequencyDomainMixer(fs=fs, f_shift_hz=5000.0)
x_mix = mixer.mix(x1, x2)

play("Mixed:", x_mix, fs)

## Inverse shift and audio export


In [ ]:
import numpy as np
from pathlib import Path
from scipy.io import wavfile
from IPython.display import Audio, display

# ----------------------------- Utilities -------------------------------------
def write_wav_int16(path: Path, fs: int, x: np.ndarray) -> None:
    """Save float signal in [-1,1] as 16-bit PCM WAV (clipped like original)."""
    x = np.clip(x, -1.0, 1.0)
    wavfile.write(str(path), fs, (x * 32767).astype(np.int16))

def fft_(x: np.ndarray) -> np.ndarray:
    return np.fft.fft(x)

def centered_roll(S: np.ndarray, k_bins: int) -> np.ndarray:
    """fftshift → roll by k → ifftshift (used for both shift and unshift)."""
    return np.fft.ifftshift(np.roll(np.fft.fftshift(S), k_bins))

# --------------------------- Core pipeline -----------------------------------
class InverseShiftProcessor:
    """
    Ensures a mixed signal exists (same logic as mixing experiment),
    then 'demodulates' by rolling the centered spectrum by -k.
    """
    def __init__(self, out_dir: Path = OUTPUT_DIR, f_shift_hz: float = 4000.0):
        self.out_dir = Path(out_dir)
        self.out_dir.mkdir(exist_ok=True)
        self.f_shift_hz = f_shift_hz
        self.fs: int | None = None
        self.k_bins: int | None = None
        self.x_mix: np.ndarray | None = None

    def ensure_mix(self, x_mix=None, fs=None, k=None) -> None:
        """
        If x_mix, fs, k are given, keep them.
        Otherwise, reproduce mixing experiment behavior to build the mixture and save it.
        """
        if x_mix is not None and fs is not None and k is not None:
            self.x_mix, self.fs, self.k_bins = np.asarray(x_mix), int(fs), int(k)
            return

        fs1, x1 = wavfile.read((DATA_DIR / "music1.wav"))
        fs2, x2 = wavfile.read((DATA_DIR / "music2.wav"))
        x1 = to_mono_float(x1)
        x2 = to_mono_float(x2)
        assert fs1 == fs2, "The sampling rate of the two files must be the same."
        self.fs = fs1
        L = min(len(x1), len(x2))
        x1 = x1[:L]
        x2 = x2[:L]

        # Compute k from desired shift
        self.k_bins = bins_for_hz(self.f_shift_hz, L, self.fs)

        # Shift S1 in the centered spectrum, sum with S2, IFFT → x_mix
        S1 = fft_(x1)
        S2 = fft_(x2)
        S1_shifted = centered_roll(S1, self.k_bins)   # fftshift->roll(+k)->ifftshift
        S_mix = S1_shifted + S2
        self.x_mix = ifft_real(S_mix)

        write_wav_int16(self.out_dir / "mix_time_b.wav", self.fs, self.x_mix)

    # ---------- Step 2: demodulate by rolling back (-k) in the centered FFT ----
    def demodulate(self) -> np.ndarray:
        """
        Compute FFT of mixture, roll centered spectrum by -k, IFFT (real).
        This mirrors the original code and reveals music1 near baseband.
        """
        assert self.x_mix is not None and self.fs is not None and self.k_bins is not None
        S_mix_f = fft_(self.x_mix)
        S_unshift = centered_roll(S_mix_f, -self.k_bins)  # fftshift->roll(-k)->ifftshift
        x_demod = ifft_real(S_unshift)

        out_path = self.out_dir / "demod_time_c.wav"
        write_wav_int16(out_path, self.fs, x_demod)
        print("Demodulated result")
        play("", x_demod, self.fs)
        print("Saved at:", out_path)
        return x_demod

# ------------------------------- Usage ---------------------------------------
proc = InverseShiftProcessor(out_dir=OUTPUT_DIR, f_shift_hz=4000.0)
proc.ensure_mix()
x_demod = proc.demodulate()